# Direct Preference Optimization (DPO) on Video Generation Preferences

Trains preference models on pairwise comparison data from human evaluations of AI-generated videos, using **CLIP visual features** extracted from the actual video frames.

**Two approaches:**
1. **Bradley-Terry (BT) Reward Model** — learns explicit reward via: $\mathcal{L} = -\log \sigma(r(y_w|x) - r(y_l|x))$
2. **DPO (Direct Preference Optimization)** — learns implicit reward: $\mathcal{L} = -\log \sigma\big(\beta \cdot (\Delta\log\pi_w - \Delta\log\pi_l)\big)$

**Reference:** Rafailov et al., *"Direct Preference Optimization: Your Language Model is Also a Reward Model"*, NeurIPS 2023.

## 0. Setup

In [ ]:
!pip install -q torch numpy matplotlib scipy huggingface_hub transformers opencv-python-headless pillow

In [ ]:
import json
import os
import time
import glob
from collections import defaultdict

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import kendalltau
from PIL import Image
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Configuration

In [ ]:
class Config:
    pref_data_path = "video_rankings3_pairwise.json"
    hf_repo = "hivamoh/wan22-physics-videos"
    dataset_dir = "./wan22-dataset"
    clip_model_name = "openai/clip-vit-base-patch32"
    n_frames = 8           # frames to sample per video
    method = "both"        # "bt", "dpo", or "both"
    beta = 0.1             # DPO temperature
    label_smoothing = 0.1
    epochs = 150
    lr = 5e-4
    wd = 5e-3
    batch_size = 128
    embed_dim = 32
    hidden_dim = 64
    dropout = 0.3
    val_ratio = 0.2
    seed = 42
    patience = 30
    log_every = 10
    output_dir = "dpo_output"
    device = device

cfg = Config()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
os.makedirs(cfg.output_dir, exist_ok=True)

## 2. Download Videos & Upload Preference Data

In [ ]:
if not os.path.exists(cfg.pref_data_path):
    try:
        from google.colab import files
        print("Upload video_rankings3_pairwise.json:")
        uploaded = files.upload()
        cfg.pref_data_path = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{cfg.pref_data_path} not found.")

print(f"Preference data: {cfg.pref_data_path}")

In [ ]:
from huggingface_hub import snapshot_download

videos_dir = os.path.join(cfg.dataset_dir, "videos")

if not os.path.isdir(videos_dir):
    print(f"Downloading videos from {cfg.hf_repo} (~1.2 GB) ...")
    snapshot_download(
        cfg.hf_repo,
        local_dir=cfg.dataset_dir,
        allow_patterns=["videos/*.mp4"],
        repo_type="dataset",
    )
    print("Download complete.")
else:
    print(f"Videos already downloaded at {videos_dir}")

n_downloaded = len(glob.glob(os.path.join(videos_dir, "*.mp4")))
print(f"Found {n_downloaded} video files.")

## 3. Extract CLIP Video Features

For each video we:
1. Sample 8 uniformly-spaced frames
2. Encode each frame with **CLIP ViT-B/32**
3. Mean-pool the frame embeddings → **(512,)** feature vector per video

In [ ]:
from transformers import CLIPModel, CLIPProcessor

cached_features_path = os.path.join(cfg.output_dir, "clip_video_features.pt")

if os.path.exists(cached_features_path):
    print("Loading cached CLIP features...")
    video_features = torch.load(cached_features_path, map_location="cpu", weights_only=True)
    feat_dim = next(iter(video_features.values())).shape[0]
    print(f"Loaded {len(video_features)} video features, dim={feat_dim}")
else:
    print(f"Loading CLIP model: {cfg.clip_model_name}")
    clip_model = CLIPModel.from_pretrained(cfg.clip_model_name).to(cfg.device).eval()
    clip_processor = CLIPProcessor.from_pretrained(cfg.clip_model_name)

    def sample_frames(video_path, n_frames=8):
        cap = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, total - 1, n_frames, dtype=int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        cap.release()
        return frames

    @torch.no_grad()
    def encode_video(video_path, n_frames=8):
        frames = sample_frames(video_path, n_frames)
        if not frames:
            return None
        inputs = clip_processor(images=frames, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(cfg.device)
        vision_out = clip_model.vision_model(pixel_values=pixel_values)
        feats = clip_model.visual_projection(vision_out.pooler_output)  # (n_frames, 512)
        feats = F.normalize(feats, dim=-1)
        return feats.mean(dim=0).cpu()  # (512,)

    # Get videos needed from preference data
    with open(cfg.pref_data_path) as f:
        raw_pref = json.load(f)
    needed_videos = set()
    for gi in raw_pref.values():
        for p in gi["pairwise_comparisons"]:
            needed_videos.update([p["video_a"], p["video_b"]])

    print(f"Extracting CLIP features for {len(needed_videos)} videos ({cfg.n_frames} frames each)...")
    video_features = {}
    missing = []
    for i, vname in enumerate(sorted(needed_videos)):
        vpath = os.path.join(videos_dir, vname)
        if os.path.exists(vpath):
            feat = encode_video(vpath, cfg.n_frames)
            if feat is not None:
                video_features[vname] = feat
            else:
                missing.append(vname)
        else:
            missing.append(vname)
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(needed_videos)} done")

    feat_dim = next(iter(video_features.values())).shape[0]
    print(f"Extracted {len(video_features)} features, dim={feat_dim}")
    if missing:
        print(f"WARNING: {len(missing)} videos could not be processed")

    torch.save(video_features, cached_features_path)
    print(f"Cached to {cached_features_path}")

    del clip_model, clip_processor
    torch.cuda.empty_cache() if cfg.device == "cuda" else None

## 4. Data Loading & Preprocessing

In [ ]:
class PairwiseDataset(Dataset):
    def __init__(self, comparisons, video_features, p2i):
        self.comps = comparisons
        self.feats = video_features
        self.p2i = p2i

    def __len__(self):
        return len(self.comps)

    def __getitem__(self, idx):
        c = self.comps[idx]
        return (
            self.p2i[c["prompt"]],
            self.feats[c["preferred"]],
            self.feats[c["rejected"]],
            abs(c["rank_a"] - c["rank_b"]),
        )


def _collate(batch):
    p, w, l, m = zip(*batch)
    return {
        "prompt": torch.tensor(p, dtype=torch.long),
        "pref_feat": torch.stack(w),
        "rej_feat": torch.stack(l),
        "margin": torch.tensor(m, dtype=torch.float),
    }


def load_preference_data(path, video_features):
    with open(path) as f:
        raw = json.load(f)
    comps, prompts = [], set()
    by_group = defaultdict(list)
    for gk, gi in raw.items():
        prompt = gi["prompt"]
        prompts.add(prompt)
        for p in gi["pairwise_comparisons"]:
            if p.get("tie"):
                continue
            if p["preferred"] not in video_features or p["rejected"] not in video_features:
                continue
            c = dict(
                prompt=prompt, preferred=p["preferred"], rejected=p["rejected"],
                video_a=p["video_a"], video_b=p["video_b"],
                rank_a=p["rank_a"], rank_b=p["rank_b"], group=gi["group"],
            )
            comps.append(c)
            by_group[gi["group"]].append(c)
    p2i = {p: i for i, p in enumerate(sorted(prompts))}
    return comps, p2i, by_group


# Groups held out from ranking (not in preference data): {0,1,8,15,21,25,27,28,45,48,50,52}
# All 48 groups in the preference JSON are training groups; we split pairs within each group.

def stratified_split(comps, val_ratio=0.2, seed=42):
    """Split pairs within each group so train and val share the same groups."""
    rng = np.random.RandomState(seed)
    groups = defaultdict(list)
    for c in comps:
        groups[c["group"]].append(c)
    train, val = [], []
    for _, cs in sorted(groups.items()):
        idx = rng.permutation(len(cs))
        n_val = max(1, int(len(cs) * val_ratio))
        val.extend(cs[i] for i in idx[:n_val])
        train.extend(cs[i] for i in idx[n_val:])
    rng.shuffle(train)
    rng.shuffle(val)
    return train, val


def ground_truth_ranks(by_group):
    gt = {}
    for gid, comps in by_group.items():
        ranks = {}
        for c in comps:
            ranks[c["video_a"]] = c["rank_a"]
            ranks[c["video_b"]] = c["rank_b"]
        gt[gid] = ranks
    return gt

In [ ]:
comps, p2i, by_group = load_preference_data(cfg.pref_data_path, video_features)
gt = ground_truth_ranks(by_group)
train_comps, val_comps = stratified_split(comps, cfg.val_ratio, cfg.seed)

n_prompts = len(p2i)

train_ds = PairwiseDataset(train_comps, video_features, p2i)
val_ds = PairwiseDataset(val_comps, video_features, p2i)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=_collate)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=_collate)

print(f"Data: {len(comps)} pairs  ({len(train_comps)} train / {len(val_comps)} val)")
print(f"      {len(video_features)} videos, {n_prompts} prompts, {len(by_group)} groups")
print(f"      CLIP feature dim = {feat_dim}")

## 5. Model Definitions

Models take **CLIP visual features** (512-d) as video representations — learned from the actual video content.

In [ ]:
def _make_scorer(feat_dim, n_prompts, embed_dim, hidden_dim, dropout):
    return nn.ModuleDict({
        "prompt_emb": nn.Embedding(n_prompts, embed_dim),
        "feat_proj": nn.Linear(feat_dim, embed_dim),
        "head": nn.Sequential(
            nn.Linear(embed_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        ),
    })


def _score(scorer, prompt_idx, video_feat):
    p = scorer["prompt_emb"](prompt_idx)
    v = scorer["feat_proj"](video_feat)
    return scorer["head"](torch.cat([p, v], dim=-1)).squeeze(-1)


class RewardModel(nn.Module):
    """Bradley-Terry reward model on CLIP video features."""

    def __init__(self, feat_dim, n_prompts, embed_dim=32, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.scorer = _make_scorer(feat_dim, n_prompts, embed_dim, hidden_dim, dropout)

    def reward(self, prompt_idx, video_feat):
        return _score(self.scorer, prompt_idx, video_feat)

    def forward(self, prompt_idx, pref_feat, rej_feat):
        return self.reward(prompt_idx, pref_feat), self.reward(prompt_idx, rej_feat)


class DPOModel(nn.Module):
    """DPO with trainable policy and frozen reference, on CLIP video features."""

    def __init__(self, feat_dim, n_prompts, embed_dim=32, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.policy = _make_scorer(feat_dim, n_prompts, embed_dim, hidden_dim, dropout)
        self.reference = _make_scorer(feat_dim, n_prompts, embed_dim, hidden_dim, dropout)
        self.reference.load_state_dict(self.policy.state_dict())
        for p in self.reference.parameters():
            p.requires_grad = False

    def forward(self, prompt_idx, pref_feat, rej_feat):
        pi_w = _score(self.policy, prompt_idx, pref_feat)
        pi_l = _score(self.policy, prompt_idx, rej_feat)
        ref_w = _score(self.reference, prompt_idx, pref_feat)
        ref_l = _score(self.reference, prompt_idx, rej_feat)
        return pi_w, pi_l, ref_w, ref_l

    def implicit_reward(self, prompt_idx, video_feat, beta):
        pi = _score(self.policy, prompt_idx, video_feat)
        ref = _score(self.reference, prompt_idx, video_feat)
        return beta * (pi - ref)

## 6. Loss Functions

In [ ]:
def bt_loss(r_w, r_l, label_smoothing=0.0):
    """Bradley-Terry with label smoothing for noisy human preferences."""
    logits = r_w - r_l
    pos = F.logsigmoid(logits)
    neg = F.logsigmoid(-logits)
    return -((1 - label_smoothing) * pos + label_smoothing * neg).mean()


def dpo_loss(pi_w, pi_l, ref_w, ref_l, beta, label_smoothing=0.0):
    """DPO loss with label smoothing."""
    logits = beta * ((pi_w - ref_w) - (pi_l - ref_l))
    pos = F.logsigmoid(logits)
    neg = F.logsigmoid(-logits)
    loss = -((1 - label_smoothing) * pos + label_smoothing * neg).mean()
    with torch.no_grad():
        acc = (logits > 0).float().mean().item()
        margin = logits.mean().item()
    return loss, {"accuracy": acc, "reward_margin": margin}

## 7. Training Loops

In [ ]:
def train_bt(train_loader, val_loader, feat_dim, n_prompts, cfg):
    print("=" * 60)
    print("  Bradley-Terry Reward Model")
    print("=" * 60)

    model = RewardModel(feat_dim, n_prompts, cfg.embed_dim, cfg.hidden_dim, cfg.dropout).to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=10)

    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc, best_state, wait = 0.0, None, 0
    t0 = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        ep_loss, ep_correct, ep_n = 0.0, 0, 0
        for b in train_loader:
            p = b["prompt"].to(cfg.device)
            wf = b["pref_feat"].to(cfg.device)
            lf = b["rej_feat"].to(cfg.device)
            r_w, r_l = model(p, wf, lf)
            loss = bt_loss(r_w, r_l, cfg.label_smoothing)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item() * len(p)
            ep_correct += (r_w > r_l).sum().item()
            ep_n += len(p)
        tl, ta = ep_loss / ep_n, ep_correct / ep_n

        model.eval()
        vl, vc, vn = 0.0, 0, 0
        with torch.no_grad():
            for b in val_loader:
                p = b["prompt"].to(cfg.device)
                wf = b["pref_feat"].to(cfg.device)
                lf = b["rej_feat"].to(cfg.device)
                r_w, r_l = model(p, wf, lf)
                vl += bt_loss(r_w, r_l, cfg.label_smoothing).item() * len(p)
                vc += (r_w > r_l).sum().item()
                vn += len(p)
        v_loss, v_acc = vl / vn, vc / vn
        sched.step(v_acc)

        hist["train_loss"].append(tl)
        hist["train_acc"].append(ta)
        hist["val_loss"].append(v_loss)
        hist["val_acc"].append(v_acc)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if epoch % cfg.log_every == 0 or epoch == 1:
            print(f"  Epoch {epoch:>4d}/{cfg.epochs}  train {tl:.4f} / {ta:.3f}  val {v_loss:.4f} / {v_acc:.3f}")

        if wait >= cfg.patience:
            print(f"  Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, os.path.join(cfg.output_dir, "best_reward_model.pt"))
    print(f"  Best val accuracy: {best_val_acc:.4f}  ({time.time() - t0:.1f}s)")
    return model, hist, best_val_acc


def train_dpo(train_loader, val_loader, feat_dim, n_prompts, cfg, beta=None):
    beta = beta or cfg.beta
    print("=" * 60)
    print(f"  DPO Model  (beta = {beta})")
    print("=" * 60)

    model = DPOModel(feat_dim, n_prompts, cfg.embed_dim, cfg.hidden_dim, cfg.dropout).to(cfg.device)
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.lr, weight_decay=cfg.wd,
    )
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=10)

    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "reward_margin": []}
    best_val_acc, best_state, wait = 0.0, None, 0
    t0 = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        ep_loss, ep_acc, ep_n = 0.0, 0.0, 0
        for b in train_loader:
            p = b["prompt"].to(cfg.device)
            wf = b["pref_feat"].to(cfg.device)
            lf = b["rej_feat"].to(cfg.device)
            pi_w, pi_l, ref_w, ref_l = model(p, wf, lf)
            loss, met = dpo_loss(pi_w, pi_l, ref_w, ref_l, beta, cfg.label_smoothing)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_([pp for pp in model.parameters() if pp.requires_grad], 1.0)
            opt.step()
            ep_loss += loss.item() * len(p)
            ep_acc += met["accuracy"] * len(p)
            ep_n += len(p)
        tl, ta = ep_loss / ep_n, ep_acc / ep_n

        model.eval()
        vl, va, vm, vn = 0.0, 0.0, [], 0
        with torch.no_grad():
            for b in val_loader:
                p = b["prompt"].to(cfg.device)
                wf = b["pref_feat"].to(cfg.device)
                lf = b["rej_feat"].to(cfg.device)
                pi_w, pi_l, ref_w, ref_l = model(p, wf, lf)
                loss, met = dpo_loss(pi_w, pi_l, ref_w, ref_l, beta, cfg.label_smoothing)
                vl += loss.item() * len(p)
                va += met["accuracy"] * len(p)
                vm.append(met["reward_margin"])
                vn += len(p)
        v_loss, v_acc, v_margin = vl / vn, va / vn, float(np.mean(vm))
        sched.step(v_acc)

        hist["train_loss"].append(tl)
        hist["train_acc"].append(ta)
        hist["val_loss"].append(v_loss)
        hist["val_acc"].append(v_acc)
        hist["reward_margin"].append(v_margin)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if epoch % cfg.log_every == 0 or epoch == 1:
            print(
                f"  Epoch {epoch:>4d}/{cfg.epochs}  train {tl:.4f} / {ta:.3f}  "
                f"val {v_loss:.4f} / {v_acc:.3f}  margin {v_margin:.3f}"
            )

        if wait >= cfg.patience:
            print(f"  Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, os.path.join(cfg.output_dir, f"best_dpo_beta{beta}.pt"))
    print(f"  Best val accuracy: {best_val_acc:.4f}  ({time.time() - t0:.1f}s)")
    return model, hist, best_val_acc

## 8. Train Models

In [ ]:
histories = {}
models = {}
best_accs = {}

if cfg.method in ("bt", "both"):
    model_bt, hist_bt, acc_bt = train_bt(train_loader, val_loader, feat_dim, n_prompts, cfg)
    histories["BT"] = hist_bt
    models["BT"] = model_bt
    best_accs["BT"] = acc_bt

if cfg.method in ("dpo", "both"):
    model_dpo, hist_dpo, acc_dpo = train_dpo(train_loader, val_loader, feat_dim, n_prompts, cfg)
    histories["DPO"] = hist_dpo
    models["DPO"] = model_dpo
    best_accs["DPO"] = acc_dpo

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, h in histories.items():
    axes[0].plot(h["train_loss"], label=f"{name} train", alpha=0.8)
    axes[0].plot(h["val_loss"], label=f"{name} val", linestyle="--", alpha=0.8)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for name, h in histories.items():
    axes[1].plot(h["train_acc"], label=f"{name} train", alpha=0.8)
    axes[1].plot(h["val_acc"], label=f"{name} val", linestyle="--", alpha=0.8)
axes[1].axhline(0.5, color="gray", linestyle=":", alpha=0.5, label="Random")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Pairwise Preference Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, "training_curves.png"), dpi=150)
plt.show()

## 10. Analysis: Accuracy by Rank-Difference Margin

In [ ]:
def accuracy_by_margin(model, data_loader, cfg, beta=None):
    model.eval()
    correct_by_margin = defaultdict(int)
    total_by_margin = defaultdict(int)
    with torch.no_grad():
        for b in data_loader:
            p = b["prompt"].to(cfg.device)
            wf = b["pref_feat"].to(cfg.device)
            lf = b["rej_feat"].to(cfg.device)
            m = b["margin"]
            if isinstance(model, RewardModel):
                r_w, r_l = model(p, wf, lf)
                pred_correct = (r_w > r_l).cpu()
            else:
                pi_w, pi_l, ref_w, ref_l = model(p, wf, lf)
                logits = (beta or cfg.beta) * ((pi_w - ref_w) - (pi_l - ref_l))
                pred_correct = (logits > 0).cpu()
            for i in range(len(m)):
                mg = int(m[i].item())
                correct_by_margin[mg] += int(pred_correct[i].item())
                total_by_margin[mg] += 1
    return {mg: correct_by_margin[mg] / total_by_margin[mg] for mg in sorted(total_by_margin.keys())}


margin_results = {}
for name, model in models.items():
    beta = cfg.beta if name == "DPO" else None
    ma = accuracy_by_margin(model, val_loader, cfg, beta=beta)
    margin_results[name] = ma
    for mg, acc in ma.items():
        print(f"  {name:>4s}  margin={mg}  acc={acc:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
all_margins = sorted(set().union(*(r.keys() for r in margin_results.values())))
x = np.arange(len(all_margins))
width = 0.8 / len(margin_results)

for i, (name, accs) in enumerate(margin_results.items()):
    vals = [accs.get(m, 0) for m in all_margins]
    ax.bar(x + i * width, vals, width, label=name, alpha=0.8)

ax.set_xticks(x + width * (len(margin_results) - 1) / 2)
ax.set_xticklabels([str(m) for m in all_margins])
ax.set_xlabel("Rank-Difference Margin")
ax.set_ylabel("Accuracy")
ax.set_title("Pairwise Accuracy by Rank-Difference Margin")
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, "margin_accuracy.png"), dpi=150)
plt.show()

## 11. Kendall τ Rank Correlation

In [ ]:
def rank_correlation(model, gt_ranks, video_features, p2i, by_group, cfg, beta=None):
    model.eval()
    taus = []
    with torch.no_grad():
        for gid, gt in gt_ranks.items():
            prompt = by_group[gid][0]["prompt"]
            pi = torch.tensor([p2i[prompt]], device=cfg.device)
            videos = [v for v in gt.keys() if v in video_features]
            gt_order = [gt[v] for v in videos]
            scores = []
            for v in videos:
                feat = video_features[v].unsqueeze(0).to(cfg.device)
                if isinstance(model, RewardModel):
                    s = model.reward(pi, feat).item()
                else:
                    s = model.implicit_reward(pi, feat, beta or cfg.beta).item()
                scores.append(s)
            tau, _ = kendalltau(gt_order, [-s for s in scores])
            if not np.isnan(tau):
                taus.append(tau)
    return float(np.mean(taus)) if taus else None, taus


print("Kendall \u03c4 Rank Correlation (vs. ground truth)")
print("=" * 50)
for name, model in models.items():
    beta = cfg.beta if name == "DPO" else None
    tau_mean, taus = rank_correlation(model, gt, video_features, p2i, by_group, cfg, beta=beta)
    if tau_mean is not None:
        print(f"  {name:>4s}  mean \u03c4 = {tau_mean:.4f}  (std = {np.std(taus):.4f})")

## 12. Summary

In [ ]:
print("=" * 50)
print("  Final Results")
print("=" * 50)
for name, acc in best_accs.items():
    print(f"  {name:>4s}  best val accuracy = {acc:.4f}")
print(f"  Random baseline              = 0.5000")
print(f"\n  Checkpoints saved to: {os.path.abspath(cfg.output_dir)}/")

## 13. Beta Sensitivity Sweep (Optional)

In [ ]:
beta_values = [0.01, 0.05, 0.1, 0.5, 1.0]
sweep_results = []

for beta in beta_values:
    model, hist, best_acc = train_dpo(train_loader, val_loader, feat_dim, n_prompts, cfg, beta=beta)
    tau_mean, _ = rank_correlation(model, gt, video_features, p2i, by_group, cfg, beta=beta)
    sweep_results.append({"beta": beta, "best_val_acc": best_acc, "kendall_tau": tau_mean})
    print(f"  beta={beta:.4f}  val_acc={best_acc:.4f}  tau={tau_mean}\n")

best_entry = max(sweep_results, key=lambda x: x["best_val_acc"])
print(f"Best beta: {best_entry['beta']}  (val acc {best_entry['best_val_acc']:.4f})")

In [ ]:
betas = [r["beta"] for r in sweep_results]
accs = [r["best_val_acc"] for r in sweep_results]
taus = [r.get("kendall_tau") for r in sweep_results]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(betas, accs, "o-", color="tab:blue", label="Val Accuracy")
ax1.set_xlabel("\u03b2 (DPO temperature)")
ax1.set_ylabel("Validation Accuracy", color="tab:blue")
ax1.set_xscale("log")
ax1.grid(True, alpha=0.3)

if any(t is not None for t in taus):
    ax2 = ax1.twinx()
    valid_b = [b for b, t in zip(betas, taus) if t is not None]
    valid_t = [t for t in taus if t is not None]
    ax2.plot(valid_b, valid_t, "s--", color="tab:orange", label="Kendall \u03c4")
    ax2.set_ylabel("Kendall \u03c4", color="tab:orange")

fig.suptitle("DPO \u03b2 Sensitivity")
plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, "beta_sweep.png"), dpi=150)
plt.show()